# AWS Strands Lab Exercise

Solution to the exercise at the end of Week 5 Day 2 (`5_agent_frameworks/2_strands_pydantic/strands_lab.ipynb`). The exercise has two parts:

1. Seed a different goal on the board, for example a short haiku about Madrid written to `madrid.txt`, and run the worker again. Does it plan sensible steps and pick the right file tools?
2. Try the one-line model swap: point the `OpenAIModel` at another OpenAI-compatible endpoint, rerun the worker, and watch the same agent run on a different model.

Run the cells top to bottom with the repo's Python 3.12 kernel. The notebook is self-contained: it keeps its own board file and its own `workspace` folder in this directory, so the lab's board and workspace are untouched.

## Setup

`board.py` lives in the day 2 folder, so instead of copying it we put that folder on `sys.path` and import it from there. `BOARD_PATH` must be set before the import: it points the board at a local `board.sqlite` in this folder, which is what keeps our runs off the lab's board.

The model is built once as an `OpenAIModel`, exactly as in the lab. Remember the lab's warning: a bare `Agent()` with no model quietly defaults to AWS Bedrock, so we always pass `model=`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# This notebook lives four levels below 5_agent_frameworks, so the day 2 folder is here:
DAY2_FOLDER = Path("../../../../2_strands_pydantic").resolve()
sys.path.insert(0, str(DAY2_FOLDER))

os.environ["BOARD_PATH"] = str(Path("board.sqlite").resolve())  # our own board, not the lab's

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

import board

load_dotenv(override=True)

MODEL = "gpt-5.4-mini"
model = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, model_id=MODEL)

## The board tools, unchanged from the lab

The three board tools are copied verbatim: `show_todos` reads the board, `plan_steps` breaks a goal into steps, `complete_task` ticks one off. In Strands each one wears the `@tool` decorator, the type hints become the argument schema, and the docstring's `Args:` section documents each parameter for the model.

In [ ]:
@tool
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board.

    Args:
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps to take, in order.
    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

@tool
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) done and record a short result summary.

    Args:
        task_id: The id of the todo to mark done.
        result: A short summary of what was accomplished.
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

## The filesystem MCP server

The same reference server as the lab, started over `npx` and scoped to a `workspace` folder inside this directory, so the agent can only touch files in there. `errlog=subprocess.DEVNULL` quiets its startup logging and is also what lets it run from a Jupyter kernel on Windows.

In [ ]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)

filesystem = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        ),
        errlog=subprocess.DEVNULL,
    ),
    startup_timeout=60
)

## Task 1: a different goal

Seed the haiku goal and let the worker run. The worker and its instruction are the same as the lab's; only the goal on the board is new.

Things to watch in the streamed trace: the worker should read the board, plan a couple of sensible steps under the goal, pick `write_file` to create `madrid.txt`, tick the steps off, and close the goal. Nobody tells it which file tool to use; it chooses from the MCP server's tool descriptions.

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

result = await worker.invoke_async("Please work the pending goal on the board.")

Now check the outcome: the board should show the goal and its steps struck through, and `madrid.txt` should hold the haiku.

In [ ]:
board.show_board()
print("\nmadrid.txt:\n" + (workspace / "madrid.txt").read_text(encoding="utf-8"))

## Task 2: the one-line model swap

Strands' headline feature: the tools, the MCP server, the instruction and the board all stay put, and only the model line changes. We point `base_url` at DeepSeek's OpenAI-compatible endpoint and set `model_id` to `deepseek-chat`. The `DEEPSEEK_API_KEY` is already in the repo-root `.env` from earlier weeks.

Any OpenAI-compatible endpoint works the same way: Groq (`https://api.groq.com/openai/v1`), a local Ollama (`http://localhost:11434/v1`), and so on. One caveat found while building this notebook: Gemini's OpenAI-compatible endpoint is not a good swap target for an agent with tools, because Gemini 3 models require a `thought_signature` field on function calls that the plain OpenAI protocol does not carry, and the agent loop fails with a 400 error on its second turn.

The goal asks for a Lisbon haiku this time, written to a different file, so both runs' outputs sit side by side in the workspace.

In [ ]:
deepseek_model = OpenAIModel(
    client_args={
        "api_key": os.environ["DEEPSEEK_API_KEY"],
        "base_url": "https://api.deepseek.com/v1",
    },
    model_id="deepseek-chat",
)

worker_on_deepseek = Agent(
    model=deepseek_model,   # the only line that changed
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Lisbon into lisbon.txt.")
board.claim_todo(goal_id)

result = await worker_on_deepseek.invoke_async("Please work the pending goal on the board.")

In [ ]:
board.show_board()
print("\nlisbon.txt:\n" + (workspace / "lisbon.txt").read_text(encoding="utf-8"))